# Arve Forecast Assessment (Simple Notebook)

This notebook does:
- load observations from `data/moi/observations`
- load forecasts from `data/moi/hydrique_ml` and `data/moi/ofev/control_member.csv`
- **exclude** OFEV `all_members.zip`
- run basic EDA plots
- compute deterministic metrics by lead time (NSE, KGE, RMSE, MAE, bias)

In [5]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

BASE = Path('..').resolve()
DATA_DIR = BASE / 'data' / 'moi'
OBS_DIR = DATA_DIR / 'observations'
OFEV_DIR = DATA_DIR / 'ofev'
ML_JSON = DATA_DIR / 'hydrique_ml' / 'Archive prévisions Hydrique ML Arve-Bout du Monde.json'
CNR_XLSX = DATA_DIR / 'sig_cnr' / 'Prévisions_QArve.xlsx'

print('BASE:', BASE)
print('DATA_DIR exists:', DATA_DIR.exists())
print('OFEV all_members exists (ignored):', (OFEV_DIR / 'all_members.zip').exists())

BASE: C:\Users\simon\Documents\DP\analysis
DATA_DIR exists: True
OFEV all_members exists (ignored): True


In [7]:
def to_naive_utc(ts_series):
    ts = pd.to_datetime(ts_series, errors='coerce', utc=True)
    return ts.dt.tz_convert(None)

def read_csv_with_fallback(path, sep=';', skiprows=0):
    encodings = ['utf-8', 'cp1252', 'latin1']
    for enc in encodings:
        try:
            return pd.read_csv(path, sep=sep, skiprows=skiprows, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError('csv', b'', 0, 1, f'Could not decode {path} with {encodings}')

def load_observations():
    # File 1: long historical file with 8 metadata lines
    f1 = OBS_DIR / '2170_Abfluss_10-Min-Mittel_1999-01-01_2024-12-31.csv'
    obs1 = read_csv_with_fallback(f1, sep=';', skiprows=8)
    obs1['datetime'] = pd.to_datetime(obs1['Zeitstempel'], errors='coerce')
    obs1['discharge_m3s'] = pd.to_numeric(obs1['Wert'], errors='coerce')
    obs1 = obs1[['datetime', 'discharge_m3s']].dropna()

    # File 2: recent observations
    f2 = OBS_DIR / 'ARVE_debit_20250101_20260216.csv'
    obs2 = read_csv_with_fallback(f2, sep=';')
    obs2['datetime'] = to_naive_utc(obs2['Timestamp_txt'])
    obs2['discharge_m3s'] = pd.to_numeric(obs2['value'], errors='coerce')
    obs2 = obs2[['datetime', 'discharge_m3s']].dropna()

    obs = pd.concat([obs1, obs2], ignore_index=True)
    obs = obs.drop_duplicates(subset='datetime').sort_values('datetime')

    # Forecasts are hourly, so aggregate obs to hourly mean
    obs_hourly = (obs
                  .set_index('datetime')
                  .resample('1h')['discharge_m3s']
                  .mean()
                  .dropna()
                  .reset_index())
    return obs_hourly

obs = load_observations()
obs.head(), obs.shape

(             datetime  discharge_m3s
 0 1999-01-01 00:00:00      24.507000
 1 1999-01-01 01:00:00      28.264000
 2 1999-01-01 02:00:00      31.589000
 3 1999-01-01 03:00:00      32.643500
 4 1999-01-01 04:00:00      29.992667,
 (237776, 2))

In [9]:
def load_hydrique_ml_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    root = raw.get('ForecastFirstDate2Timeseries', {})
    rows = []

    for forecast_date, ts_map in root.items():
        fdt = pd.to_datetime(forecast_date, errors='coerce')
        if pd.isna(fdt):
            continue
        for valid_time, value in ts_map.items():
            vdt = pd.to_datetime(valid_time, errors='coerce')
            if pd.isna(vdt):
                continue
            lead_h = (vdt - fdt).total_seconds() / 3600.0
            rows.append({
                'forecast_date': fdt,
                'datetime': vdt,
                'lead_time_h': lead_h,
                'discharge_m3s': float(value),
                'model': 'HYDRIQUE_ML'
            })

    df = pd.DataFrame(rows)
    return df

def load_ofev_control(path):
    df = pd.read_csv(path)
    df['forecast_date'] = pd.to_datetime(df['forecast_date'], errors='coerce')
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df['lead_time_h'] = pd.to_numeric(df['lead_time_h'], errors='coerce')
    df['discharge_m3s'] = pd.to_numeric(df['discharge_m3s'], errors='coerce')
    keep = ['forecast_date', 'datetime', 'lead_time_h', 'discharge_m3s', 'model']
    return df[keep].dropna()

def load_sig_cnr(path):
    # Simple flexible parser for CNR Excel
    xls = pd.ExcelFile(path)
    best = None

    for sheet in xls.sheet_names:
        df0 = pd.read_excel(path, sheet_name=sheet)
        if df0.empty:
            continue

        df = df0.copy()
        col_map = {c: str(c).strip().lower() for c in df.columns}

        # datetime candidates
        dt_candidates = [
            c for c, lc in col_map.items()
            if ('date' in lc) or ('time' in lc) or ('heure' in lc) or ('valid' in lc)
        ]
        if not dt_candidates:
            dt_candidates = list(df.columns)

        dt_col = None
        best_dt_score = -1
        for c in dt_candidates:
            s = pd.to_datetime(df[c], errors='coerce')
            score = s.notna().mean()
            if score > best_dt_score:
                best_dt_score = score
                dt_col = c

        # value candidates
        val_candidates = [
            c for c, lc in col_map.items()
            if ('debit' in lc) or ('flow' in lc) or ('discharge' in lc) or ('q' == lc) or ('q_' in lc) or ('m3' in lc)
        ]
        if not val_candidates:
            val_candidates = list(df.columns)

        val_col = None
        best_val_score = -1
        for c in val_candidates:
            s = pd.to_numeric(df[c], errors='coerce')
            score = s.notna().mean()
            if score > best_val_score:
                best_val_score = score
                val_col = c

        if dt_col is None or val_col is None:
            continue

        out = pd.DataFrame({
            'datetime': pd.to_datetime(df[dt_col], errors='coerce'),
            'discharge_m3s': pd.to_numeric(df[val_col], errors='coerce')
        }).dropna()

        # Optional columns
        fdate_col = None
        lead_col = None
        model_col = None
        for c, lc in col_map.items():
            if ('forecast' in lc and 'date' in lc) or ('run' in lc and 'date' in lc) or ('initial' in lc):
                fdate_col = c
            if ('lead' in lc) or ('echeance' in lc) or ('horizon' in lc):
                lead_col = c
            if 'model' in lc:
                model_col = c

        if fdate_col is not None:
            out['forecast_date'] = pd.to_datetime(df[fdate_col], errors='coerce')
        else:
            out['forecast_date'] = out['datetime']

        if lead_col is not None:
            out['lead_time_h'] = pd.to_numeric(df[lead_col], errors='coerce')
        else:
            out['lead_time_h'] = (out['datetime'] - out['forecast_date']).dt.total_seconds() / 3600.0

        if model_col is not None:
            out['model'] = df[model_col].astype(str)
        else:
            out['model'] = 'SIG_CNR'

        out = out[['forecast_date', 'datetime', 'lead_time_h', 'discharge_m3s', 'model']].dropna()

        # keep the best parsed sheet (most rows)
        if best is None or len(out) > len(best):
            best = out

    if best is None:
        return pd.DataFrame(columns=['forecast_date', 'datetime', 'lead_time_h', 'discharge_m3s', 'model'])

    return best

fc_ml = load_hydrique_ml_json(ML_JSON)
fc_ofev = load_ofev_control(OFEV_DIR / 'control_member.csv')
fc_cnr = load_sig_cnr(CNR_XLSX)

# Keep all models except OFEV all_members (zip is intentionally ignored)
forecasts = pd.concat([fc_ml, fc_ofev, fc_cnr], ignore_index=True)
forecasts = forecasts[~forecasts['model'].astype(str).str.contains('all_members', case=False, na=False)]
forecasts = forecasts.dropna().sort_values(['model', 'forecast_date', 'datetime'])

print('Rows by model:')
print(forecasts['model'].value_counts().head(10))
forecasts.head()

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [10]:
# Join forecast values to observed hourly discharge
eval_df = forecasts.merge(obs, on='datetime', how='inner', suffixes=('_sim', '_obs'))
eval_df = eval_df.rename(columns={
    'discharge_m3s_sim': 'q_sim',
    'discharge_m3s_obs': 'q_obs'
})

print('Rows for evaluation:', len(eval_df))
eval_df[['datetime', 'forecast_date', 'lead_time_h', 'model', 'q_obs', 'q_sim']].head()

Rows for evaluation: 2603452


,datetime,forecast_date,lead_time_h,model,q_obs,q_sim
0,2020-09-15 06:00:00,2020-09-15 06:00:00,0.0,C1E,51.815333,379.41
1,2020-09-15 06:00:00,2020-09-15 06:00:00,0.0,C1E,51.815333,43.20
2,2020-09-15 06:00:00,2020-09-15 06:00:00,0.0,C1E,51.815333,43.20
3,2020-09-15 06:00:00,2020-09-15 06:00:00,0.0,C1E,51.815333,379.40
4,2020-09-15 06:00:00,2020-09-15 06:00:00,0.0,C1E,51.815333,54.00


In [ ]:
# EDA 1: full observed hourly series
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(obs['datetime'], obs['discharge_m3s'], lw=0.8, color='black')
ax.set_title('Observed discharge at Geneva Bout du Monde (hourly)')
ax.set_ylabel('m3/s')
ax.set_xlabel('time')
plt.show()

In [ ]:
# EDA 2: lead-time overlay for a selected forecast run
LEADS_TO_SHOW = [6, 24, 48, 72]

# choose one forecast initialization per model (first available)
picked = (forecasts[['model', 'forecast_date']]
          .drop_duplicates()
          .sort_values('forecast_date')
          .groupby('model', as_index=False)
          .first())

fig, ax = plt.subplots(figsize=(14, 5))

# plot observations around selected windows
obs_plot = obs.set_index('datetime')

for _, r in picked.iterrows():
    sub = forecasts[(forecasts['model'] == r['model']) & (forecasts['forecast_date'] == r['forecast_date'])].copy()
    sub = sub[sub['lead_time_h'].round().isin(LEADS_TO_SHOW)]
    ax.plot(sub['datetime'], sub['discharge_m3s'], marker='o', ms=3, lw=1, label=f"{r['model']} (selected leads)")

# observed over full x-range used by selected forecasts
if not picked.empty:
    tmin = forecasts['datetime'].min()
    tmax = forecasts['datetime'].max()
    o = obs[(obs['datetime'] >= tmin) & (obs['datetime'] <= tmax)]
    ax.plot(o['datetime'], o['discharge_m3s'], color='black', lw=2, alpha=0.4, label='OBS')

ax.set_title('Representative forecast points at lead +6/+24/+48/+72')
ax.set_ylabel('m3/s')
ax.legend()
plt.show()

In [ ]:
# Deterministic metrics
def nse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    den = np.sum((y_true - np.mean(y_true)) ** 2)
    if den == 0:
        return np.nan
    return 1 - np.sum((y_true - y_pred) ** 2) / den

def kge_components(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return np.nan, np.nan, np.nan, np.nan
    r = np.corrcoef(y_true, y_pred)[0, 1]
    alpha = np.std(y_pred) / np.std(y_true) if np.std(y_true) != 0 else np.nan
    beta = np.mean(y_pred) / np.mean(y_true) if np.mean(y_true) != 0 else np.nan
    kge = 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)
    return kge, r, alpha, beta

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))

def bias(y_true, y_pred):
    return np.mean(np.asarray(y_pred) - np.asarray(y_true))

rows = []
group_cols = ['model', 'lead_time_h']

for (model, lead), g in eval_df.groupby(group_cols):
    y = g['q_obs'].values
    p = g['q_sim'].values
    kge_v, r, alpha, beta_v = kge_components(y, p)
    rows.append({
        'model': model,
        'lead_time_h': float(lead),
        'n': len(g),
        'NSE': nse(y, p),
        'KGE': kge_v,
        'r': r,
        'alpha': alpha,
        'beta': beta_v,
        'RMSE': rmse(y, p),
        'MAE': mae(y, p),
        'Bias': bias(y, p)
    })

metrics_by_lead = pd.DataFrame(rows).sort_values(['model', 'lead_time_h'])
metrics_by_lead.head(20)

In [ ]:
# Skill degradation and bias profile
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)

for model, g in metrics_by_lead.groupby('model'):
    g = g.sort_values('lead_time_h')
    axes[0].plot(g['lead_time_h'], g['NSE'], label=model)
    axes[1].plot(g['lead_time_h'], g['RMSE'], label=model)
    axes[2].plot(g['lead_time_h'], g['Bias'], label=model)

axes[0].set_title('NSE vs lead time')
axes[1].set_title('RMSE vs lead time')
axes[2].set_title('Bias vs lead time')
for ax in axes:
    ax.set_xlabel('lead time (h)')
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('NSE')
axes[1].set_ylabel('RMSE (m3/s)')
axes[2].set_ylabel('Bias (m3/s)')
axes[2].legend()
plt.tight_layout()
plt.show()

metrics_by_lead.to_csv(BASE / 'metrics_by_lead.csv', index=False)
print('Saved:', BASE / 'metrics_by_lead.csv')

## Optional next simple additions
- Event analysis: define threshold (e.g. 200 m3/s) and compute peak error / timing error per event.
- Split by period (COSMO vs ICON) before comparing lead-time metrics.
- Add flow-duration curves for each model on matched timestamps.